# Feature 1 - The Transaction Parser

In [21]:
from ast import unparse
import pandas as pd
df=pd.read_csv('Data set for DADS June.csv')
# df=pd.read_csv('Paytm_UPI.csv')
df.columns=df.columns.str.lower()
print(df.columns)
def run_transacton():
  total_rows=len(df)
  clean_amount=(
      df['amount'].astype(str).str.replace("₹", "", regex=False)
        .str.replace("Rs.", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
  )
  df['amount']=pd.to_numeric(clean_amount,errors='coerce')
  unparseable_amount=int(df['amount'].isna().sum())
  df['date']=pd.to_datetime(df['date'],dayfirst=True,format='mixed',errors='coerce')
  unparseable_date=int(df['date'].isna().sum())
  df['type']=df['type'].astype(str).str.strip().str.lower()
  df_cleaned=df.drop_duplicates()
  dropped_duplicates=total_rows-len(df_cleaned)
  valid_dates=df_cleaned['date'].dropna()
  total_months=0
  if not valid_dates.empty:
    years_difference=valid_dates.dt.year.max()-valid_dates.dt.year.min()
    months_dfference=(
        valid_dates.dt.month.max()-valid_dates.dt.month.min()
    )
    total_months=(years_difference*12)+months_dfference+1
  print(f"Parsed {len(df_cleaned)} transactions across {total_months} months. ")
  print(f"Dropped {dropped_duplicates} duplicates. ")
  print(f"{unparseable_amount} unparseable amounts, {unparseable_date} unparseable dates.")

run_transacton()

Index(['date', 'time', 'description', 'type', 'amount', 'balance', 'mode',
       'ref'],
      dtype='object')
Parsed 1310 transactions across 6 months. 
Dropped 18 duplicates. 
0 unparseable amounts, 0 unparseable dates.


# Feature 2 - Vendor Extractor

In [22]:
# print(df['description'].unique())
vendor_rule={
    "Swiggy": ["swiggy", "bundl", "instamart"],
        "Zomato": ["zomato", "dineout"],
        "Starbucks": ["starbucks", "tata starbucks"],
        "Third Wave Coffee": ["thirdwave", "third wave coffee", "twc india"],
        "Cafe Coffee Day": ["ccd", "cafe coffee day", "coffee day"],
        "Truffles": ["truffles"],"Empire Restaurant": ["empire restaurant"],
        "Meghana Foods": ["meghana foods"],"Blinkit": ["blinkit", "grofers"],
        "Zepto": ["zepto", "kiranakart"],
        "BigBasket": ["bigbasket", "innovative retail"],
        "DMart": ["dmart", "avenue supermarts"],"Amazon": ["amazon", "amzn"],
        "Flipkart": ["flipkart", "fkart"],
        "Myntra": ["myntra"],
        "Nykaa": ["nykaa", "fsn e-commerce"],"Uber": ["uber"],
        "Ola": ["ola ride", "olacabs", "ola prime", "ola cabs", "ola electric"],
        "Rapido": ["rapido", "roppen transportation"],
        "BMTC Public Transport": ["bmtc", "tummoc"],
       "Netflix": ["netflix"],
        "Spotify": ["spotify"],
        "BookMyShow": ["bookmyshow", "bms movie", "bigtree entertainment"],
        "Disney Hotstar": ["hotstar", "star india"],
        "Airtel": ["airtel", "bharti airtel"],
        "Jio": ["jio"],
        "Vodafone Idea": ["vodafone", "vi postpaid", "vi-recharge"],
        "BESCOM Electricity": ["bescom", "bangalore elec supply"],
        "BWSSB Water Bill": ["bwssb"],
        "HP Petrol Pump": ["hp petrol"],
        "BPCL Petrol Pump": ["bpcl"],
        "IOCL Petrol Pump": ["indian oil", "ioc"],
        # 7. Salary, Investments & Rent
        "Startup Salary": ["techcrush"],
        "Zerodha": ["zerodha", "groww", "nextbillion"],
        "House Rent": ["rent-landlord"],
        # 8. Structural Financial Fallbacks (Processed after explicit vendors)
        "Cash Withdrawal": ["atm-wdl"],
        "Generic Restaurant": ["restaurant"],
        "P2P Transfer": ["upi-aman", "upi-ankit", "upi-karan", "upi-neha", "upi-priya", "upi-sneha", "upi-vikas"]
}
def resolve_canonical(description):
  desc_lower=str(description).lower()
  for canonical,keywords in vendor_rule.items():
     for keyword in keywords:
       if keyword in desc_lower:
        return canonical
  if "upi-" in desc_lower:
    return "P@P Transfer"
  return "Other"
df['vendor_clean']=df['description'].apply(resolve_canonical)
print("canonical vendors:",df['vendor_clean'].nunique())
# print(df['vendor_clean'])
print(df['vendor_clean'].value_counts().head(10))

canonical vendors: 39
vendor_clean
Swiggy       248
Zomato       134
Amazon        87
Zepto         73
Uber          71
Blinkit       56
Rapido        55
Ola           51
Flipkart      48
Starbucks     42
Name: count, dtype: int64


# Feature 3 - Category Tagger

In [23]:

# print(df['vendor_clean'].unique())
vendor_category={
    # Core Consumption Categories
        "Swiggy": "Food Delivery",
        "Zomato": "Food Delivery",
        "Blinkit": "Quick Commerce",
        "Zepto": "Quick Commerce",
        "BigBasket": "Groceries",
        "DMart": "Groceries",
        "Amazon": "E-commerce",
        "Flipkart": "E-commerce",
        "Myntra": "E-commerce",
        "Nykaa": "E-commerce",
        # Transport & Commute
        "Uber": "Transport",
        "Ola": "Transport",
        "Rapido": "Transport",
        "BMTC Public Transport": "Transport",
        # Cafe & Social Eating
        "Starbucks": "Cafe",
        "Third Wave Coffee": "Cafe",
        "Cafe Coffee Day": "Cafe",
        "Truffles": "Restaurants",
        "Empire Restaurant": "Restaurants",
        "Meghana Foods": "Restaurants",
        "Generic Restaurant": "Restaurants",
        # Entertainment & Digital Subscriptions
        "Netflix": "Subscriptions",
        "Spotify": "Subscriptions",
        "BookMyShow": "Entertainment",
        "Disney Hotstar": "Subscriptions",
        # Utilities & Infrastructure
        "Airtel": "Utilities",
        "Jio": "Utilities",
        "Vodafone Idea": "Utilities",
        "BESCOM Electricity": "Utilities",
        "BWSSB Water Bill": "Utilities",
        # Fuel, Investments & Structured Inflows/Outflows
        "HP Petrol Pump": "Fuel",
        "BPCL Petrol Pump": "Fuel",
        "IOCL Petrol Pump": "Fuel",
        "Zerodha": "Investments",
        "Startup Salary": "Investments",  # Kept here as it relates to core asset tracking
        "House Rent": "Utilities",
        # Structural Fallbacks
        "Cash Withdrawal": "Cash Withdrawal",
        "P2P Transfer": "Personal Transfer",

}
df['category']=df['vendor_clean'].map(vendor_category)
print(df['category'].value_counts())


category
Food Delivery        382
Transport            216
E-commerce           174
Quick Commerce       129
Cafe                  99
Restaurants           63
Utilities             50
Groceries             41
Subscriptions         31
Investments           29
Fuel                  29
Personal Transfer     19
Cash Withdrawal       17
Entertainment         13
Name: count, dtype: int64


# Feature 4 - Spending Overview

In [24]:
# df['type'].unique()
credit_df=df['type']=='cr'
debit_df=df['type'].isin(['dr','debit'])
total_credit=df.loc[credit_df,'amount'].sum()
total_debit=df.loc[debit_df,'amount'].sum()
net_savings=total_credit-total_debit
net_savings_percentage=(net_savings/total_credit)*100
debit=df[debit_df]
top_categories=(debit.groupby('category')['amount']
                .sum().sort_values(ascending=False))
top_vendors=(debit.groupby('vendor_clean')['amount']
                .sum().sort_values(ascending=False))
print(f"total transections:{len(df)}")
print(f"Total credit: {total_credit:,.2f}")
print(f"Total debit: {total_debit:,.2f}")
print(f"Net savings: {net_savings}%")
print(f"Top 5 Spending Categories:\n{top_categories.head(5)}")
print(f"Top 5 Spending Vendors:\n{top_vendors.head(5)}")

total transections:1328
Total credit: 509,774.00
Total debit: 1,729,708.00
Net savings: -1219934.0%
Top 5 Spending Categories:
category
E-commerce       638210.0
Investments      248160.0
Food Delivery    183637.0
Utilities        152245.0
Restaurants       97912.0
Name: amount, dtype: float64
Top 5 Spending Vendors:
vendor_clean
Amazon        348447.0
Zerodha       248160.0
Flipkart      191926.0
House Rent    108000.0
Swiggy        106758.0
Name: amount, dtype: float64


# Feature 5 - Monthly Trend Analysis

In [25]:
import numpy as np
import pandas as pd
df_cleaned=df
debit_mask=df['type'].isin(['dr','debit'])
debit_df=df[debit_mask].copy()
debit_df['month']=debit_df['date'].dt.month
debit_df['year']=debit_df['date'].dt.year
month_pivot=debit_df.pivot_table(
    values='amount',index='category',columns='month',aggfunc='sum'
)
month_pivot=month_pivot.fillna(0)
month_names={1:'jan',2:'feb',3:'mar',4:'apr',5:'may',6:'jun',7:'jul',8:'aug',9:'sep',10:'oct',11:'nov',12:'dec'}
month_pivot=month_pivot.rename(columns=month_names)
month_first=month_pivot['jan']
month_last=month_pivot['jun']
month_diff=(month_last-month_first)/(month_first+1)*100
sorted_growth=month_diff.sort_values(ascending=False)
biggest_growth_cat=sorted_growth.idxmax()
biggest_growth_val=sorted_growth.max()
biggest_decline_cat=sorted_growth.idxmin()
biggest_decline_val=sorted_growth.min()
print(f"Biggest Growth Category: {biggest_growth_cat} ({biggest_growth_val}%)")
print(f"Biggest Decline Category: {biggest_decline_cat} ({biggest_decline_val}%)")
mont_growth=month_pivot.pct_change(axis=1)*100
print(mont_growth.fillna(0))


Biggest Growth Category: Cash Withdrawal (749.6251874062968%)
Biggest Decline Category: Fuel (-92.32979487588794%)
month              jan         feb          mar        apr         may  \
category                                                                 
Cafe               0.0   15.799458    27.498245  20.484581  -13.650213   
Cash Withdrawal    0.0  150.000000    60.000000 -31.250000   45.454545   
E-commerce         0.0   -4.676394    36.294689 -45.978366   59.193285   
Entertainment      0.0  -62.470309   410.126582  -8.023160 -100.000000   
Food Delivery      0.0   26.519791    -4.216887  12.956619   -9.869982   
Fuel               0.0  -94.468685  1158.489658 -28.458951  -51.180682   
Groceries          0.0  -51.436342   -26.624665  86.802353  -17.279537   
Investments        0.0  -60.970025   357.626667 -21.149700  -10.157780   
Personal Transfer  0.0  -45.427916    -0.233372 -84.491228  414.630468   
Quick Commerce     0.0   26.939132    -1.261854 -26.155991   19.561569 

# Feature 6 - Time-of-Day Patterns

In [26]:
debit_df=df[df['type'].isin(['dr','debit'])].copy()
debit_df['hour']=debit_df['time'].str[:2].astype(int)
hour_matrix=debit_df.pivot_table(
    values='amount',index='category',columns='hour',aggfunc='sum'
)
hour_matrix=hour_matrix.fillna(0)
print("Hourly Spending Patterns:",hour_matrix)
for category in hour_matrix.index:
  row =hour_matrix.loc[category]
  total=row.sum()
  if total==0:
    continue
  peak_hour=row.idxmax()
  peak_amount=row.max()
  peak_percentage=(peak_amount/total)*100
  print(f"Peak Hour : {peak_hour:02d}:00")
  print(f"Peak Spend: Rs. {peak_amount:,.0f}")
  print(f"Share     : {peak_percentage:.1f}%")
  max_amt=row.max()
  for hour in range(24):
    amount=row.get(hour,0)
    if max_amt==0:
      bar=""
    else:
      bar="#"*int((amount/max_amt)*20)
    print(f"{hour:02d}:00 | {bar} ({amount:,.0f})")
food=debit_df[debit_df['category']=='Food Delivery']
if len(food)>0:
  late_orders=food[
      (food['hour']>=21)| (food['hour']<=1)
  ]
percent=(len(late_orders)/len(food))*100
print(f"Late Orders: {len(late_orders)} ({percent:.1f}%)")
cafe=debit_df[debit_df['category']=='Cafe']
if len(cafe)>0:
  morning_orders=cafe[(cafe['hour']>=6)&(cafe['hour']<=11)]
  percent=(len(morning_orders)/len(cafe))*100
  print(f"Morning Orders: {len(morning_orders)} ({percent:.1f}")


Hourly Spending Patterns: hour                    0        1        2        3        4        5   \
category                                                                  
Cafe                 872.0    549.0      0.0    316.0      0.0      0.0   
Cash Withdrawal        0.0      0.0      0.0      0.0      0.0      0.0   
E-commerce         18650.0   9801.0  10246.0  14865.0  12018.0  17258.0   
Entertainment        562.0    620.0      0.0      0.0      0.0      0.0   
Food Delivery       1515.0   3017.0   1374.0   3849.0   2618.0   3488.0   
Fuel                2675.0   2048.0      0.0    676.0   2010.0   2105.0   
Groceries           3895.0   1536.0   2352.0  10064.0   1169.0      0.0   
Investments         4883.0  15000.0      0.0      0.0  18834.0   4496.0   
Personal Transfer      0.0      0.0      0.0      0.0      0.0      0.0   
Quick Commerce       859.0   1402.0    697.0    732.0    768.0   1448.0   
Restaurants         1411.0    960.0   1336.0      0.0      0.0   2126.0   

# Feature 7 - Anomaly Detection

In [27]:
debit_df=df[df['type'].isin(['dr','debit'])].copy()
debit_df['mean']=(
    debit_df.groupby('category')['amount']
    .transform('mean')
)
debit_df['std']=(
    debit_df.groupby('category')['amount']
    .transform('std')
)
debit_df['std']=debit_df['std'].replace(0,np.nan)
debit_df['z_score']=(debit_df['amount']-debit_df['mean'])/debit_df['std']
anomalies=debit_df[debit_df['z_score']>2]
anomalies=anomalies.sort_values(by='z_score',ascending=False)
print("top 5 anomalous transaction")
for _,row in anomalies.head(5).iterrows():
  print(f"Date : {row['date']}")
  print(f"Category : {row['category']}")
  print(f"Description : {row['description']}")
  print(f"Amount : {row['amount']}")
  print(f"Mean : {row['mean']}")
  print(f"Standard Deviation : {row['std']}")
  print(f"Z-Score : {row['z_score']}")
print("total anomalies:",len(anomalies))

top 5 anomalous transaction
Date : 2024-06-22 00:00:00
Category : Food Delivery
Description : POS DINEOUT
Amount : 7935.0
Mean : 480.72513089005236
Standard Deviation : 437.1052899301162
Z-Score : 17.053728336945376
Date : 2024-02-26 00:00:00
Category : Restaurants
Description : POS BANGALORE RESTAURANT
Amount : 8383.0
Mean : 1554.1587301587301
Standard Deviation : 1686.4960258969197
Z-Score : 4.049129772606209
Date : 2024-06-26 00:00:00
Category : E-commerce
Description : AMAZONIN MARKETPLACE
Amount : 22008.0
Mean : 3667.8735632183907
Standard Deviation : 4736.212246758909
Z-Score : 3.872319372792499
Date : 2024-02-07 00:00:00
Category : E-commerce
Description : UPI-AMAZONPAY@HDFCBANK
Amount : 21986.0
Mean : 3667.8735632183907
Standard Deviation : 4736.212246758909
Z-Score : 3.8676743106935496
Date : 2024-03-31 00:00:00
Category : Restaurants
Description : POS MEGHANA FOODS
Amount : 7931.0
Mean : 1554.1587301587301
Standard Deviation : 1686.4960258969197
Z-Score : 3.7811184680674894
t

# Feature 8 - Spending Archetype Detection

In [33]:
total_debit=debit_df['amount'].sum()
category_spend=debit_df.groupby('category')['amount'].sum()
food_total=(
    category_spend.get('Food Delivery',0)+
    category_spend.get('Restaurants', 0)
    + category_spend.get('Cafe', 0)
)
print("\nSPENDING ARCHETYPES\n")
food_percent=(food_total/total_debit)*100
if food_percent>5:
  print(f"THE FOODIE ({food_percent:.1f}%) of total")
quick=category_spend.get('Quick Commerce',0)
quick_percent=(quick/total_debit)*100
if quick_percent>5:
  print(f"THE QUICK COMMER  JUNKIE ({quick_percent:.1f}%) of total")
shop=category_spend.get('Groceries',0)
shop_percent=(shop/total_debit)*100
if shop_percent>5:
  print(f"THE SHOPAHOLIC ({shop_percent:.1f}%) of total")
invest=category_spend.get('Investments',0)
invest_percent=(invest/total_debit)*100
if invest_percent>5:
  print(f"THE INVESTOR ({invest_percent:.1f}%) of total")
food_orders=debit_df[debit_df['category']=='Food Delivery'].copy()
food_orders['hour']=food_orders['time'].str[:2].astype(int)
late_orders=food_orders[(food_orders['hour']>=21)| (food_orders['hour']<=2)]
late_orders_percent=(len(late_orders)/len(food_orders))*100
if late_orders_percent>5:
  print(f"THE LATE-NIGHT SNAKER ({late_orders_percent:.1f}%) of total")
transport=category_spend.get('Transport',0)
transport_percent=(transport/total_debit)*100
if transport_percent>5:
  print(f"THE CAB COMMUTER ({transport_percent:.1f}%) of total")
subscription_count=debit_df[debit_df['category']=='Subscriptions']['vendor_clean'].nunique()
if subscription_count>5:
  print(f"THE SUBSCRIPTION LOVER ({subscription_count:.1f}%) of total")
savings_rate=((total_credit-total_debit)/total_credit)*100
if savings_rate<10:
  print(f"THE YOLO SPENDER (Savings Rate = {savings_rate:.1f}%)")

if savings_rate>40:
  print(f"THE DISCIPLINED SAVER (Savings Rate = {savings_rate:.1f}%)")


SPENDING ARCHETYPES

THE FOODIE (18.1%) of total
THE INVESTOR (14.3%) of total
THE LATE-NIGHT SNAKER (21.2%) of total
THE YOLO SPENDER (Savings Rate = -239.3%)


# Final output

In [29]:
df['type']=df['type'].str.strip().str.lower()
debit_df=df[df['type'].isin(['dr','debit'])].copy()
credit_df=df[df['type'].isin(['cr','credit'])].copy()
print("="*46)
print("SpendDNA REPORT - RAHUL SHARMA")
df['month']=df['date'].dt.month
num_months=df['month'].nunique()
print(f" {num_months} months ",end="-")

total_msg=len(df)
print(f"{total_msg} transactions",end="-")
start_month=df['date'].min().strftime("%B")
end_month=df['date'].max().strftime("%B %Y")
print(f" {start_month} - {end_month}")


print("="*46)
print("EXECUTIVE SUMMARY")

total_debit=debit_df['amount'].sum()
print("total debits:",total_debit)
total_credit=credit_df['amount'].sum()
print("total_credits:",total_credit)
net_changes=total_credit-total_debit
if net_changes>0:
  print("Netchanges :",net_changes,end="(normal speending)")
elif net_changes==0:
  print("Netchanges :",net_changes,end="(no changes)")
else:
  print("Netchanges :",net_changes,end="(overspending)")
print()
total_credit-total_debit
savings_rate=((total_credit-total_debit)/total_credit)*100
if savings_rate<0:
  print("Savings rate:",savings_rate,end="(BURNING SAVINGS)")
elif savings_rate==0:
  print("Savings rate:",savings_rate,end="(NO SAVINGS)")
else:
  print("Savings rate:",savings_rate,end="(flowering SAVINGS)")
print("\n\nTOP CATEGORIES (% of debit total)")
print()
debit_total=debit_df['amount'].sum()
cat_amount=debit_df.groupby('category')['amount'].sum()
top_categories=cat_amount.sort_values(ascending=False)
for category,amount in top_categories.items():
  percent=(amount/debit_total)*100
  bar='#'*int(percent)
  print(f"{category} :{bar}  {percent:.1f}% Rs.{amount:.1f} ")
print("\n TOP VENDORS\n")
top_vendors=debit_df.groupby('vendor_clean')['amount'].sum().sort_values(ascending=False).head(10)
vendor_orders=debit_df.groupby('vendor_clean').size()
for ven,amo in top_vendors.items():
  order=vendor_orders.get(ven,0)
  print(f"{ven} Rs. {amo:.1f}  ({order} orders)")

print("\nTIME-OF-DAY PATTERNS\n")
debit_df['hour']=debit_df['time'].str[:2].astype(int)

food = debit_df[debit_df['category'] == 'Food Delivery']
if len(food)>0:
  late_food=food[(food['hour']>=21)|(food['hour']<=1)]
  late_percent=(len(late_food)/len(food))*100
  print(f"Food delivery peaks: {len(late_food)} ({late_percent:.1f}%)")
cafe=debit_df[debit_df['category']=='Cafe']
if len(cafe)>0:
  morning_orders=cafe[(cafe['hour']>=6)&(cafe['hour']<=11)]
  moring_percent=(len(morning_orders)/len(cafe))*100
  print(f"Cafe peaks: {len(morning_orders)} ({moring_percent:.1f}% of orders) ")
quick=debit_df[debit_df['category']=='Quick Commerce']
if len(quick)>0:
 hour_count=quick.groupby('hour').size()
 max_orders=hour_count.max()
 min_orders=hour_count.min()
 if max_orders-min_orders<=20:
  print("Quick Commerce : Evenly distributed")
 else:
  peak_hour=hour_count.idxmax()
  print(f"Quick Commerce :Peaks around {peak_hour:02d}:00")
import calendar
print("\nMONTHLY TREND (Food Delivery)\n")
food_df=debit_df[debit_df['category'] == 'Food Delivery'].copy()
food_df['month']=food_df['date'].dt.month
monthly=food_df.groupby('month')['amount'].sum()
max_amount=monthly.max()
for month,amt in monthly.items():
  month_name=calendar.month_abbr[month]
  bar_len=int((amt/max_amount)*15)
  print(f"{month_name} Rs.{amt:.1f} {'#'*bar_len} ")
debit_df['mean']=debit_df.groupby('category')['amount'].transform('mean')
debit_df['std']=debit_df.groupby('category')['amount'].transform('std')
debit_df['std']=debit_df['std'].replace(0,np.nan)
debit_df['z_score']=(debit_df['amount']-debit_df['mean'])/debit_df['std']
anomalies=debit_df[debit_df['z_score']>3]
anomalies=anomalies.sort_values(by='z_score',ascending=False)

print("\nTOP ANOMALIES (3+ stddev from category mean)\n")
for _,row in anomalies.iterrows():
  date=row['date'].strftime("%d %b")
  print(f"{date}  - {row['vendor_clean']}  Rs.{row['amount']:,.0f}  ({row['z_score']:.1f})")
print("\nRAHUL'S SPENDING ARCHETYPES\n")
category_spend=debit_df.groupby('category')['amount'].sum()
food_total=(category_spend.get('Food Delivery',0)+category_spend.get('Restaurants',0)+category_spend.get('Cafe',0))
# df['category'].unique()
food_df = debit_df[debit_df['category'] == 'Food Delivery'].copy()
food_df['hour'] = food_df['time'].str[:2].astype(int)
late_orders = food_df[(food_df['hour'] >= 21) | (food_df['hour'] <= 1)]
late_pct = (len(late_orders) / len(food_df)) * 100
food_pct=(food_total/debit_total)*100
quick_pct=(category_spend.get('Quick Commerce',0)/debit_total)*100
shop_pct=(category_spend.get('E-commerce',0)/debit_total)*100
invest_pct=(category_spend.get('Investments',0)/debit_total)*100

if food_pct>10:
  print(f"-> THE FOODIE ({food_pct:.1f}% on food)")
if quick_pct>5:
  print(f"-> THE QUICK COMMERCE ({quick_pct:.1f}% on Q-com)")
if shop_pct>5:
  print(f"-> THE SHOPAHOLIC ({shop_pct:.1f}% on e-commerce)")
if invest_pct>55:
  print(f"-> THE INVESTOR ({invest_pct:.1f}% on investments)")
if late_pct>5:
  print(f"-> THE LATE-NIGHT SNAKER ({late_pct:.1f}% on food)")
if savings_rate<10:
  print(f"-> THE YOLO SPENDER ({savings_rate:.1f}% savings)")


SpendDNA REPORT - RAHUL SHARMA
 6 months -1328 transactions- January - June 2024
EXECUTIVE SUMMARY
total debits: 1729708.0
total_credits: 509774.0
Netchanges : -1219934.0(overspending)
Savings rate: -239.30879173908437(BURNING SAVINGS)

TOP CATEGORIES (% of debit total)

E-commerce :####################################  36.9% Rs.638210.0 
Investments :##############  14.3% Rs.248160.0 
Food Delivery :##########  10.6% Rs.183637.0 
Utilities :########  8.8% Rs.152245.0 
Restaurants :#####  5.7% Rs.97912.0 
Fuel :#####  5.6% Rs.96567.0 
Quick Commerce :###  3.9% Rs.66619.0 
Groceries :###  3.4% Rs.59407.0 
Transport :##  2.7% Rs.46062.0 
Cash Withdrawal :##  2.6% Rs.45500.0 
Cafe :#  1.8% Rs.31445.0 
Personal Transfer :#  1.5% Rs.25698.0 
Subscriptions :#  1.1% Rs.18471.0 
Entertainment :  0.5% Rs.8293.0 

 TOP VENDORS

Amazon Rs. 348447.0  (87 orders)
Zerodha Rs. 248160.0  (23 orders)
Flipkart Rs. 191926.0  (48 orders)
House Rent Rs. 108000.0  (6 orders)
Swiggy Rs. 106758.0  (248 orders